In [0]:
# Notebook: utils/spark_config
# Purpose: Central ADLS authentication config via Azure Key Vault RBAC
# Usage: "%run ./utils/spark_config" at top of every notebook
# Never run standalone — always called via %run

# ── Fetch credentials from Key Vault via RBAC backed secret scope ──
client_id      = dbutils.secrets.get("portfolio-scope", "sp-client-id")
client_secret  = dbutils.secrets.get("portfolio-scope", "sp-client-secret")
tenant_id      = dbutils.secrets.get("portfolio-scope", "sp-tenant-id")
storage_acct   = dbutils.secrets.get("portfolio-scope", "storage-account-name")

# ── Configure Spark OAuth for ADLS Gen2 ───────────────────────────
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_acct}.dfs.core.windows.net",
    "OAuth")
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_acct}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_acct}.dfs.core.windows.net",
    client_id)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_acct}.dfs.core.windows.net",
    client_secret)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_acct}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# ── Centralised container paths ────────────────────────────────────
paths = {
    "staging"  : f"abfss://staging@{storage_acct}.dfs.core.windows.net/",
    "bronze"   : f"abfss://bronze@{storage_acct}.dfs.core.windows.net/",
    "silver"   : f"abfss://silver@{storage_acct}.dfs.core.windows.net/",
    "gold"     : f"abfss://gold@{storage_acct}.dfs.core.windows.net/",
    "control"  : f"abfss://control@{storage_acct}.dfs.core.windows.net/"
}

# ── Specific table and checkpoint paths ───────────────────────────
table_paths = {
    "bronze_transactions" : paths["bronze"] + "transactions/",
    "silver_transactions" : paths["silver"] + "transactions/",
    "gold_daily_summary"  : paths["gold"]   + "daily_summary/",
    "gold_fraud_metrics"  : paths["gold"]   + "fraud_metrics/",
    "control_simulator"   : paths["control"]+ "simulator_state/",
    "bronze_checkpoint"   : paths["bronze"] + "_checkpoints/transactions/",
    "staging_master"      : paths["staging"]+ "master/",
    "staging_daily"       : paths["staging"]+ "daily/"
}

print(f"✅ Spark config loaded via Azure Key Vault RBAC")
print(f"✅ Storage account : {storage_acct}")
print(f"✅ Paths initialised: {list(paths.keys())}")